# Dataset Preprocessing


This notebook applies the preprocessing pipeline to the datasets, preparing the data for modeling.

In [1]:
import sys
import os

# Add the "src" directory to sys.path for module imports
sys.path.append(os.path.abspath(os.path.join(os.pardir, "src")))

In [2]:
# Main libraries for preprocessing and splitting
import pandas as pd
from sklearn.model_selection import train_test_split
from preprocessing import preprocess_dataframe

## Suicide and Depression Detection Dataset

### 1. Data Loading

In [3]:
# Load Suicide and Depression Detection dataset
df1 = pd.read_csv("../data/raw/suicide-and-depression-detection/raw.csv")

### 2. Text Preprocessing

In [4]:
# Apply preprocessing to the original DataFrame
df_clean_ds1 = preprocess_dataframe(df1, text_column="text", min_length=4)
# Show the first 10 cleaned examples
df_clean_ds1.head(10)

,Unnamed: 0,text,class,clean_text,text_length
0,2,Ex Wife Threatening SuicideRecently I left my ...,suicide,ex wife threatening suiciderecently i left my ...,143
1,3,Am I weird I don't get affected by compliments...,non-suicide,am i weird i don't get affected by compliments...,27
2,4,Finally 2020 is almost over... So I can never ...,non-suicide,finally 2020 is almost over... so i can never ...,26
3,8,i need helpjust help me im crying so hard,suicide,i need helpjust help me im crying so hard,9
4,9,"I’m so lostHello, my name is Adam (16) and I’v...",suicide,"i’m so losthello, my name is adam (16) and i’v...",438
5,11,Honetly idkI dont know what im even doing here...,suicide,honetly idki dont know what im even doing here...,291
6,12,[Trigger warning] Excuse for self inflicted bu...,suicide,[trigger warning] excuse for self inflicted bu...,243
7,13,It ends tonight.I can’t do it anymore. \nI quit.,suicide,it ends tonight.i can’t do it anymore. i quit.,9
8,16,"Everyone wants to be ""edgy"" and it's making me...",non-suicide,"everyone wants to be ""edgy"" and it's making me...",195
9,18,My life is over at 20 years oldHello all. I am...,suicide,my life is over at 20 years oldhello all. i am...,224


### 3. Selection and Renaming of Relevant Columns

In [5]:
# Select and rename relevant columns from the cleaned DataFrame
df_selected_ds1 = df_clean_ds1[["class", "clean_text", "text_length"]].copy()
df_selected_ds1 = df_selected_ds1.rename(columns={"class": "label"})
# Show the first 10 examples
df_selected_ds1.head(10)

,label,clean_text,text_length
0,suicide,ex wife threatening suiciderecently i left my ...,143
1,non-suicide,am i weird i don't get affected by compliments...,27
2,non-suicide,finally 2020 is almost over... so i can never ...,26
3,suicide,i need helpjust help me im crying so hard,9
4,suicide,"i’m so losthello, my name is adam (16) and i’v...",438
5,suicide,honetly idki dont know what im even doing here...,291
6,suicide,[trigger warning] excuse for self inflicted bu...,243
7,suicide,it ends tonight.i can’t do it anymore. i quit.,9
8,non-suicide,"everyone wants to be ""edgy"" and it's making me...",195
9,suicide,my life is over at 20 years oldhello all. i am...,224


In [6]:
# Standardize target column values
df_selected_ds1["label"] = df_selected_ds1["label"].replace({"non-suicide": "not_suicide"})
# Show the first 10 examples after standardization
df_selected_ds1.head(10)

,label,clean_text,text_length
0,suicide,ex wife threatening suiciderecently i left my ...,143
1,not_suicide,am i weird i don't get affected by compliments...,27
2,not_suicide,finally 2020 is almost over... so i can never ...,26
3,suicide,i need helpjust help me im crying so hard,9
4,suicide,"i’m so losthello, my name is adam (16) and i’v...",438
5,suicide,honetly idki dont know what im even doing here...,291
6,suicide,[trigger warning] excuse for self inflicted bu...,243
7,suicide,it ends tonight.i can’t do it anymore. i quit.,9
8,not_suicide,"everyone wants to be ""edgy"" and it's making me...",195
9,suicide,my life is over at 20 years oldhello all. i am...,224


### 4. Class Analysis and Balancing

In [7]:
# Review class balance before balancing
print("Original class distribution:")
print(df_selected_ds1["label"].value_counts())

Original class distribution:
label
suicide        115803
not_suicide    115580
Name: count, dtype: int64


In [8]:
# Balance classes by undersampling the majority class
min_count_ds1 = df_selected_ds1["label"].value_counts().min()
df_balanced_ds1 = (
    df_selected_ds1.groupby("label", group_keys=False)
    .sample(n=min_count_ds1, random_state=42)
    .reset_index(drop=True)
)

In [9]:
# Review class balance after balancing
print("\nClass distribution after balancing:")
print(df_balanced_ds1["label"].value_counts())


Class distribution after balancing:
label
not_suicide    115580
suicide        115580
Name: count, dtype: int64


### 5. Train, Validation, and Test Split

In [10]:
# Split the dataset into train, validation, and test sets (80/10/10, stratified)
train_df1, temp_df1 = train_test_split(
    df_balanced_ds1, test_size=0.2, stratify=df_balanced_ds1["label"], random_state=42
)
validation_df1, test_df1 = train_test_split(
    temp_df1, test_size=0.5, stratify=temp_df1["label"], random_state=42
)
print(f"\nSizes: train={len(train_df1)}, val={len(validation_df1)}, test={len(test_df1)}")


Sizes: train=184928, val=23116, test=23116


### 6. Saving Processed Data

In [11]:
# Ensure the target folder exists for saving processed data
os.makedirs("../data/processed/suicide-and-depression-detection", exist_ok=True)

In [12]:
# Save the cleaned DataFrame to CSV
df_selected_ds1.to_csv("../data/processed/suicide-and-depression-detection/clean.csv", index=False)
print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [13]:
# Save train, validation, and test splits to CSV
train_df1.to_csv("../data/processed/suicide-and-depression-detection/train.csv", index=False)
validation_df1.to_csv("../data/processed/suicide-and-depression-detection/validation.csv", index=False)
test_df1.to_csv("../data/processed/suicide-and-depression-detection/test.csv", index=False)
print("Splits saved successfully!")

Splits saved successfully!


### 7. Final Remarks


- Preprocessing ensured text cleaning, duplicate removal, and elimination of very short examples, making the dataset more consistent for modeling.
- Class balancing was performed by undersampling, ensuring an equal number of examples for each class and avoiding bias during training.
- Stratified splitting into train, validation, and test sets (80/10/10) preserved class proportions in each split, allowing for more reliable model evaluation.
- Processed files and splits were saved in appropriate directories, ready for use in the next steps of fine-tuning and experimentation with language models.
- The developed pipeline is compatible with Transformer/BERT models, retaining relevant information in the texts and avoiding excessive removals, which is essential for good performance in text classification tasks.

## Sentiment Analysis for Mental Health Dataset

### 1. Data Loading

In [14]:
# Load Sentiment Analysis for Mental Health dataset
df2 = pd.read_csv("../data/raw/sentiment-analysis-for-mental-health/raw.csv")

### 2. Text Preprocessing

In [15]:
# Apply preprocessing to the original DataFrame
df_clean_ds2 = preprocess_dataframe(df2, text_column="statement", min_length=1)
# Show the first 10 cleaned examples
df_clean_ds2.head(10)

,Unnamed: 0,statement,status,clean_text,text_length
0,0,oh my gosh,Anxiety,oh my gosh,3
1,1,"trouble sleeping, confused mind, restless hear...",Anxiety,"trouble sleeping, confused mind, restless hear...",10
2,2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety,"all wrong, back off dear, forward doubt. stay ...",14
3,3,I've shifted my focus to something else but I'...,Anxiety,i've shifted my focus to something else but i'...,11
4,4,"I'm restless and restless, it's been a month n...",Anxiety,"i'm restless and restless, it's been a month n...",14
5,5,"every break, you must be nervous, like somethi...",Anxiety,"every break, you must be nervous, like somethi...",14
6,6,"I feel scared, anxious, what can I do? And may...",Anxiety,"i feel scared, anxious, what can i do? and may...",17
7,7,Have you ever felt nervous but didn't know why?,Anxiety,have you ever felt nervous but didn't know why?,9
8,8,"I haven't slept well for 2 days, it's like I'm...",Anxiety,"i haven't slept well for 2 days, it's like i'm...",14
9,9,"I'm really worried, I want to cry.",Anxiety,"i'm really worried, i want to cry.",7


### 3. Selection and Renaming of Relevant Columns

In [16]:
# Select and rename relevant columns from the cleaned DataFrame
df_selected_ds2 = df_clean_ds2[["status", "clean_text", "text_length"]].copy()
df_selected_ds2 = df_selected_ds2.rename(columns={"status": "label"})
# Show the first 10 examples
df_selected_ds2.head(10)

,label,clean_text,text_length
0,Anxiety,oh my gosh,3
1,Anxiety,"trouble sleeping, confused mind, restless hear...",10
2,Anxiety,"all wrong, back off dear, forward doubt. stay ...",14
3,Anxiety,i've shifted my focus to something else but i'...,11
4,Anxiety,"i'm restless and restless, it's been a month n...",14
5,Anxiety,"every break, you must be nervous, like somethi...",14
6,Anxiety,"i feel scared, anxious, what can i do? and may...",17
7,Anxiety,have you ever felt nervous but didn't know why?,9
8,Anxiety,"i haven't slept well for 2 days, it's like i'm...",14
9,Anxiety,"i'm really worried, i want to cry.",7


In [17]:
# Standardize target column values
df_selected_ds2["label"] = df_selected_ds2["label"].replace({
    "Normal": "normal",
    "Depression": "depression",
    "Suicidal": "suicide",
    "Anxiety": "anxiety",
    "Bipolar": "bipolar",
    "Stress": "stress",
    "Personality disorder": "personality_disorder",
})
# Show the first 10 examples after standardization
df_selected_ds2.head(10)

,label,clean_text,text_length
0,anxiety,oh my gosh,3
1,anxiety,"trouble sleeping, confused mind, restless hear...",10
2,anxiety,"all wrong, back off dear, forward doubt. stay ...",14
3,anxiety,i've shifted my focus to something else but i'...,11
4,anxiety,"i'm restless and restless, it's been a month n...",14
5,anxiety,"every break, you must be nervous, like somethi...",14
6,anxiety,"i feel scared, anxious, what can i do? and may...",17
7,anxiety,have you ever felt nervous but didn't know why?,9
8,anxiety,"i haven't slept well for 2 days, it's like i'm...",14
9,anxiety,"i'm really worried, i want to cry.",7


### 4. Class Analysis and Balancing

In [18]:
# Review class balance before balancing
print("Original class distribution:")
print(df_selected_ds2["label"].value_counts())

Original class distribution:
label
normal                  16005
depression              15085
suicide                 10634
anxiety                  3611
bipolar                  2501
stress                   2289
personality_disorder      894
Name: count, dtype: int64


### 5. Train, Validation, and Test Split

In [19]:
# Split the dataset into train, validation, and test sets (80/10/10, stratified)
train_df2, temp_df2 = train_test_split(
    df_selected_ds2, test_size=0.2, stratify=df_selected_ds2["label"], random_state=42
)
validation_df2, test_df2 = train_test_split(
    temp_df2, test_size=0.5, stratify=temp_df2["label"], random_state=42
)
print(f"\nSizes: train={len(train_df2)}, val={len(validation_df2)}, test={len(test_df2)}")


Sizes: train=40815, val=5102, test=5102


### 6. Saving Processed Data

In [20]:
# Ensure the target folder exists for saving processed data
os.makedirs("../data/processed/sentiment-analysis-for-mental-health", exist_ok=True)

In [21]:
# Save the cleaned DataFrame to CSV
df_selected_ds2.to_csv("../data/processed/sentiment-analysis-for-mental-health/clean.csv", index=False)
print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [22]:
# Save train, validation, and test splits to CSV
train_df2.to_csv("../data/processed/sentiment-analysis-for-mental-health/train.csv", index=False)
validation_df2.to_csv("../data/processed/sentiment-analysis-for-mental-health/validation.csv", index=False)
test_df2.to_csv("../data/processed/sentiment-analysis-for-mental-health/test.csv", index=False)
print("Splits saved successfully!")

Splits saved successfully!


### 7. Final Remarks


- Preprocessing ensured text cleaning, duplicate removal, and elimination of very short examples, making the dataset more consistent for modeling.
- Only examples with text length equal to or greater than 1 word were kept, ensuring that only minimally informative texts were used.
- Due to the low number of examples in some classes, class balancing was not performed by undersampling at this stage. If necessary during modeling, additional techniques such as oversampling may be applied to address imbalance.
- Stratified splitting into train, validation, and test sets (80/10/10) preserved class proportions in each split, allowing for more reliable model evaluation.
- Processed files and splits were saved in appropriate directories, ready for use in the next steps of fine-tuning and experimentation with language models.
- The developed pipeline is compatible with Transformer/BERT models, retaining relevant information in the texts and avoiding excessive removals, which is essential for good performance in text classification tasks.

## Sentimental Analysis for Tweets Dataset

### 1. Data Loading

In [23]:
# Load Sentimental Analysis for Tweets dataset
df3 = pd.read_csv("../data/raw/sentimental-analysis-for-tweets/raw.csv")

### 2. Text Preprocessing

In [24]:
# Apply preprocessing to the original DataFrame
df_clean_ds3 = preprocess_dataframe(df3, text_column="message to examine", min_length=4)
# Show the first 10 cleaned examples
df_clean_ds3.head(10)

,Index,message to examine,label (depression result),clean_text,text_length
0,106,just had a real good moment. i missssssssss hi...,0,just had a real good moment. i missssssssss hi...,11
1,288,@lapcat Need to send 'em to my accountant tomo...,0,need to send 'em to my accountant tomorrow. od...,21
2,540,ADD ME ON MYSPACE!!! myspace.com/LookThunder,0,add me on myspace!!! myspace.com/lookthunder,5
3,624,so sleepy. good times tonight though,0,so sleepy. good times tonight though,6
4,701,"@SilkCharm re: #nbn as someone already said, d...",0,"re: as someone already said, does fiber to the...",19
5,808,23 or 24ï¿½C possible today. Nice,0,23 or 24ï¿½c possible today. nice,6
6,1193,nite twitterville workout in the am -ciao,0,nite twitterville workout in the am -ciao,7
7,1324,"@daNanner Night, darlin'! Sweet dreams to you",0,"night, darlin'! sweet dreams to you",6
8,1368,Finally! I just created my WordPress Blog. The...,0,finally! i just created my wordpress blog. the...,18
9,1578,kisha they cnt get over u til they get out frm...,0,kisha they cnt get over u til they get out frm...,18


### 3. Selection and Renaming of Relevant Columns

In [25]:
# Select and rename relevant columns from the cleaned DataFrame
df_selected_ds3 = df_clean_ds3[["label (depression result)", "clean_text", "text_length"]].copy()
df_selected_ds3 = df_selected_ds3.rename(columns={"label (depression result)": "label"})
# Show the first 10 examples
df_selected_ds3.head(10)

,label,clean_text,text_length
0,0,just had a real good moment. i missssssssss hi...,11
1,0,need to send 'em to my accountant tomorrow. od...,21
2,0,add me on myspace!!! myspace.com/lookthunder,5
3,0,so sleepy. good times tonight though,6
4,0,"re: as someone already said, does fiber to the...",19
5,0,23 or 24ï¿½c possible today. nice,6
6,0,nite twitterville workout in the am -ciao,7
7,0,"night, darlin'! sweet dreams to you",6
8,0,finally! i just created my wordpress blog. the...,18
9,0,kisha they cnt get over u til they get out frm...,18


In [26]:
# Standardize target column values
df_selected_ds3["label"] = df_selected_ds3["label"].replace({0: "not_depression", 1: "depression"})
# Show the first 10 examples after standardization
df_selected_ds3.head(10)

,label,clean_text,text_length
0,not_depression,just had a real good moment. i missssssssss hi...,11
1,not_depression,need to send 'em to my accountant tomorrow. od...,21
2,not_depression,add me on myspace!!! myspace.com/lookthunder,5
3,not_depression,so sleepy. good times tonight though,6
4,not_depression,"re: as someone already said, does fiber to the...",19
5,not_depression,23 or 24ï¿½c possible today. nice,6
6,not_depression,nite twitterville workout in the am -ciao,7
7,not_depression,"night, darlin'! sweet dreams to you",6
8,not_depression,finally! i just created my wordpress blog. the...,18
9,not_depression,kisha they cnt get over u til they get out frm...,18


### 4. Class Analysis and Balancing

In [27]:
# Review class balance before balancing
print("Original class distribution:")
print(df_selected_ds3["label"].value_counts())

Original class distribution:
label
not_depression    7305
depression        2207
Name: count, dtype: int64


In [28]:
# Balance classes by undersampling the majority class
min_count_ds3 = df_selected_ds3["label"].value_counts().min()
df_balanced_ds3 = (
    df_selected_ds3.groupby("label", group_keys=False)
    .sample(n=min_count_ds3, random_state=42)
    .reset_index(drop=True)
)

In [29]:
# Review class balance after balancing
print("\nClass distribution after balancing:")
print(df_balanced_ds3["label"].value_counts())


Class distribution after balancing:
label
depression        2207
not_depression    2207
Name: count, dtype: int64


### 5. Train, Validation, and Test Split

In [30]:
# Split the dataset into train, validation, and test sets (80/10/10, stratified)
train_df3, temp_df3 = train_test_split(
    df_balanced_ds3, test_size=0.2, stratify=df_balanced_ds3["label"], random_state=42
)
validation_df3, test_df3 = train_test_split(
    temp_df3, test_size=0.5, stratify=temp_df3["label"], random_state=42
)
print(f"\nSizes: train={len(train_df3)}, val={len(validation_df3)}, test={len(test_df3)}")


Sizes: train=3531, val=441, test=442


### 6. Saving Processed Data

In [31]:
# Ensure the target folder exists for saving processed data
os.makedirs("../data/processed/sentimental-analysis-for-tweets", exist_ok=True)

In [32]:
# Save the cleaned DataFrame to CSV
df_selected_ds3.to_csv("../data/processed/sentimental-analysis-for-tweets/clean.csv", index=False)
print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [33]:
# Save train, validation, and test splits to CSV
train_df3.to_csv("../data/processed/sentimental-analysis-for-tweets/train.csv", index=False)
validation_df3.to_csv("../data/processed/sentimental-analysis-for-tweets/validation.csv", index=False)
test_df3.to_csv("../data/processed/sentimental-analysis-for-tweets/test.csv", index=False)
print("Splits saved successfully!")

Splits saved successfully!


### 7. Final Remarks


- Preprocessing ensured text cleaning, duplicate removal, and elimination of very short examples, making the dataset more consistent for modeling.
- Class balancing was performed by undersampling, ensuring an equal number of examples for each class and avoiding bias during training.
- Stratified splitting into train, validation, and test sets (80/10/10) preserved class proportions in each split, allowing for more reliable model evaluation.
- Processed files and splits were saved in appropriate directories, ready for use in the next steps of fine-tuning and experimentation with language models.
- The developed pipeline is compatible with Transformer/BERT models, retaining relevant information in the texts and avoiding excessive removals, which is essential for good performance in text classification tasks.

## Mental Health Corpus Dataset

### 1. Data Loading

In [34]:
# Load Mental Health Corpus dataset
df4 = pd.read_csv("../data/raw/mental-health-corpus/raw.csv")

### 2. Text Preprocessing

In [35]:
# Apply preprocessing to the original DataFrame
df_clean_ds4 = preprocess_dataframe(df4, text_column="text", min_length=4)
# Show the first 10 cleaned examples
df_clean_ds4.head(10)

,text,label,clean_text,text_length
0,dear american teens question dutch person hear...,0,dear american teens question dutch person hear...,23
1,nothing look forward lifei dont many reasons k...,1,nothing look forward lifei dont many reasons k...,20
2,music recommendations im looking expand playli...,0,music recommendations im looking expand playli...,64
3,im done trying feel betterthe reason im still ...,1,im done trying feel betterthe reason im still ...,100
4,worried year old girl subject domestic physic...,1,worried year old girl subject domestic physica...,311
5,hey rredflag sure right place post this goes ...,1,hey rredflag sure right place post this goes i...,61
6,feel like someone needs hear tonight feeling r...,0,feel like someone needs hear tonight feeling r...,79
7,deserve liveif died right noone would carei re...,1,deserve liveif died right noone would carei re...,51
8,feels good ive set dateim killing friday nice ...,1,feels good ive set dateim killing friday nice ...,14
9,live guiltok made stupid random choice its ge...,1,live guiltok made stupid random choice its get...,66


### 3. Selection and Renaming of Relevant Columns

In [36]:
# Select relevant columns from the cleaned DataFrame
df_selected_ds4 = df_clean_ds4[["label", "clean_text", "text_length"]].copy()
# Show the first 10 examples
df_selected_ds4.head(10)

,label,clean_text,text_length
0,0,dear american teens question dutch person hear...,23
1,1,nothing look forward lifei dont many reasons k...,20
2,0,music recommendations im looking expand playli...,64
3,1,im done trying feel betterthe reason im still ...,100
4,1,worried year old girl subject domestic physica...,311
5,1,hey rredflag sure right place post this goes i...,61
6,0,feel like someone needs hear tonight feeling r...,79
7,1,deserve liveif died right noone would carei re...,51
8,1,feels good ive set dateim killing friday nice ...,14
9,1,live guiltok made stupid random choice its get...,66


In [37]:
# Standardize target column values
df_selected_ds4["label"] = df_selected_ds4["label"].replace({0: "not_poisonous", 1: "poisonous"})
# Show the first 10 examples after standardization
df_selected_ds4.head(10)

,label,clean_text,text_length
0,not_poisonous,dear american teens question dutch person hear...,23
1,poisonous,nothing look forward lifei dont many reasons k...,20
2,not_poisonous,music recommendations im looking expand playli...,64
3,poisonous,im done trying feel betterthe reason im still ...,100
4,poisonous,worried year old girl subject domestic physica...,311
5,poisonous,hey rredflag sure right place post this goes i...,61
6,not_poisonous,feel like someone needs hear tonight feeling r...,79
7,poisonous,deserve liveif died right noone would carei re...,51
8,poisonous,feels good ive set dateim killing friday nice ...,14
9,poisonous,live guiltok made stupid random choice its get...,66


### 4. Class Analysis and Balancing

In [38]:
# Review class balance before balancing
print("Original class distribution:")
print(df_selected_ds4["label"].value_counts())

Original class distribution:
label
not_poisonous    13954
poisonous        13752
Name: count, dtype: int64


In [39]:
# Balance classes by undersampling the majority class
min_count_ds4 = df_selected_ds4["label"].value_counts().min()
df_balanced_ds4 = (
    df_selected_ds4.groupby("label", group_keys=False)
    .sample(n=min_count_ds4, random_state=42)
    .reset_index(drop=True)
)

In [40]:
# Review class balance after balancing
print("\nClass distribution after balancing:")
print(df_balanced_ds4["label"].value_counts())


Class distribution after balancing:
label
not_poisonous    13752
poisonous        13752
Name: count, dtype: int64


### 5. Train, Validation, and Test Split

In [41]:
# Split the dataset into train, validation, and test sets (80/10/10, stratified)
train_df4, temp_df4 = train_test_split(
    df_balanced_ds4, test_size=0.2, stratify=df_balanced_ds4["label"], random_state=42
)
validation_df4, test_df4 = train_test_split(
    temp_df4, test_size=0.5, stratify=temp_df4["label"], random_state=42
)
print(f"\nSizes: train={len(train_df4)}, val={len(validation_df4)}, test={len(test_df4)}")


Sizes: train=22003, val=2750, test=2751


### 6. Saving Processed Data

In [42]:
# Ensure the target folder exists for saving processed data
os.makedirs("../data/processed/mental-health-corpus", exist_ok=True)

In [43]:
# Save the cleaned DataFrame to CSV
df_selected_ds4.to_csv("../data/processed/mental-health-corpus/clean.csv", index=False)
print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [44]:
# Save train, validation, and test splits to CSV
train_df4.to_csv("../data/processed/mental-health-corpus/train.csv", index=False)
validation_df4.to_csv("../data/processed/mental-health-corpus/validation.csv", index=False)
test_df4.to_csv("../data/processed/mental-health-corpus/test.csv", index=False)
print("Splits saved successfully!")

Splits saved successfully!


### 7. Final Remarks


- Preprocessing ensured text cleaning, duplicate removal, and elimination of very short examples, making the dataset more consistent and suitable for modeling.
- Class balancing was performed by undersampling, ensuring an equal number of examples for each class and reducing the risk of bias during model training.
- Stratified splitting into train, validation, and test sets (80/10/10) preserved class proportions in each split, allowing for more reliable and comparable model evaluation.
- Processed files and splits were saved in appropriate directories, facilitating organization and reuse of data in the next project steps.
- The developed pipeline is compatible with Transformer/BERT models, retaining relevant information in the texts and avoiding excessive removals, which is essential for good performance in text classification tasks.

# Next Steps


- Start fine-tuning Transformer/BERT models using the prepared train, validation, and test sets.